# P1 Retained Base Tables

This notebook implements **Phase P1** from the frozen pipeline.

## Goals

- load the February 2026 GSTP utility-scale main sheet
- standardize key categorical and numeric fields
- retain the four approved statuses
- apply the baseline country filter `record count >= 30`
- assign the frozen main size bins
- export the retained analytical base tables required for downstream phases
- save QA/QC audit tables needed to defend `T1` and `SI.1`

## Required outputs

- `artifacts/tables/p1_retained_sample_summary.csv`
- `artifacts/tables/p1_country_scope_summary.csv`
- `artifacts/tables/p1_mode_activity_summary.csv`
- `artifacts/reports/p1_retained_scope_report.md`

## Additional QA/QC outputs

- `artifacts/tables/p1_status_scope_summary.csv`
- `artifacts/tables/p1_country_filter_audit.csv`
- `artifacts/tables/p1_bin_assignment_audit.csv`

This notebook also saves a reproducibility manifest and an analysis-ready retained baseline table for downstream use.


In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'data' / 'Global-Solar-Power-Tracker-February-2026.xlsx').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'Global-Solar-Power-Tracker-February-2026.xlsx'
TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

DATASET_RELEASE = 'Global-Solar-Power-Tracker-February-2026.xlsx'
SHEET_NAME = 'Utility-Scale (1 MW+)'
PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P1__nogit'
STATUS_ORDER = ['operating', 'construction', 'pre-construction', 'announced']
SIZE_BIN_LABELS = ['1-5', '5-50', '50-500', '500+']
SIZE_BINS = [1, 5, 50, 500, np.inf]
COUNTRY_THRESHOLD = 30
MODE_ORDER = [f'{status} | {label}' for status in STATUS_ORDER for label in SIZE_BIN_LABELS]

manifest = {
    'notebook': 'P1_Retained_Base_Tables.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'dataset_release': DATASET_RELEASE,
    'sheet_name': SHEET_NAME,
    'retained_statuses': STATUS_ORDER,
    'country_filter_rule': f'record_count >= {COUNTRY_THRESHOLD}',
    'size_bins': SIZE_BIN_LABELS,
    'git_shortsha': 'nogit',
}
(RUNTIME / 'p1_retained_scope_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown(f"**Data file:** `{DATA_PATH.name}`"))


## Load and Standardize the Utility-Scale Main Sheet

This phase keeps the cleaning logic intentionally narrow. We standardize only the fields needed for the frozen retained-scope build and do not introduce any analytical choices beyond the contract.


In [ ]:
raw_util = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, engine='openpyxl')

util = raw_util.copy()
util['Country/Area'] = util['Country/Area'].astype('string').str.strip()
util['Status'] = util['Status'].astype('string').str.strip().str.lower()
util['Capacity (MW)'] = pd.to_numeric(util['Capacity (MW)'], errors='coerce')
util['Start year'] = pd.to_numeric(util['Start year'], errors='coerce')
util['Retired year'] = pd.to_numeric(util['Retired year'], errors='coerce')
util['Date Last Researched'] = pd.to_datetime(util['Date Last Researched'], errors='coerce')
util['Location accuracy'] = util['Location accuracy'].astype('string').str.strip().str.lower()
util['Phase Name'] = util['Phase Name'].fillna('--').astype(str).str.strip()
util['Country/Area'] = util['Country/Area'].fillna('Unknown').astype(str)

main_status_mask = util['Status'].isin(STATUS_ORDER)
capacity_mask = util['Capacity (MW)'].notna() & util['Capacity (MW)'].ge(1)
clean_util = util.loc[main_status_mask & capacity_mask].copy()

cleaning_summary = pd.DataFrame([
    {'metric': 'raw_rows_utility_sheet', 'value': int(len(util))},
    {'metric': 'kept_rows_after_status_and_capacity_filters', 'value': int(len(clean_util))},
    {'metric': 'rows_dropped_non_core_status', 'value': int((~main_status_mask).sum())},
    {'metric': 'rows_dropped_missing_or_lt1mw', 'value': int((~capacity_mask).sum())},
    {'metric': 'countries_after_basic_cleaning', 'value': int(clean_util['Country/Area'].nunique())},
])

display(cleaning_summary)
display(clean_util[['Country/Area', 'Status', 'Capacity (MW)', 'Start year', 'Phase Name']].head(10))


## Apply the Frozen Retained-Scope Rules

This step implements the P1 analytical freeze exactly as specified:

- statuses: `operating`, `construction`, `pre-construction`, `announced`
- country filter: `record count >= 30`
- size bins: `1-5`, `5-50`, `50-500`, `500+`


In [ ]:
country_record_counts = clean_util.groupby('Country/Area').size().rename('record_count').sort_values(ascending=False)
selected_countries = country_record_counts[country_record_counts >= COUNTRY_THRESHOLD].index

retained = clean_util.loc[clean_util['Country/Area'].isin(selected_countries)].copy()
retained['capacity_bin'] = pd.cut(
    retained['Capacity (MW)'],
    bins=SIZE_BINS,
    labels=SIZE_BIN_LABELS,
    right=False,
    include_lowest=True,
)
retained = retained.loc[retained['capacity_bin'].notna()].copy()
retained['mode'] = pd.Categorical(
    retained['Status'].astype(str) + ' | ' + retained['capacity_bin'].astype(str),
    categories=MODE_ORDER,
    ordered=True,
)
retained['pipeline_version'] = PIPELINE_VERSION
retained['run_id'] = RUN_ID

retained_country_counts = retained.groupby('Country/Area').size().rename('record_count')
retained_country_capacity = retained.groupby('Country/Area')['Capacity (MW)'].sum().rename('capacity_mw')
retained_total_capacity = float(retained['Capacity (MW)'].sum())

scope_snapshot = pd.Series({
    'countries_retained': int(retained['Country/Area'].nunique()),
    'rows_retained': int(len(retained)),
    'capacity_mw_retained': round(retained_total_capacity, 1),
    'assigned_modes': int(retained['mode'].nunique()),
    'median_rows_per_country': float(retained_country_counts.median()),
    'mean_rows_per_country': round(float(retained_country_counts.mean()), 2),
})

display(scope_snapshot.to_frame('value'))


## Build and Export the Required P1 Tables

The tables below are the formal P1 outputs referenced by the pipeline. They are saved to `artifacts/tables/` and `artifacts/reports/` for direct reuse in later phases.


In [ ]:
status_scope_summary = (
    util.assign(
        capacity_mw_for_sum=util['Capacity (MW)'].fillna(0),
        status_group=np.where(util['Status'].isin(STATUS_ORDER), 'retained', 'excluded'),
        capacity_ge_1=np.where(util['Capacity (MW)'].ge(1).fillna(False), 1, 0),
    )
    .groupby(['Status', 'status_group'], as_index=False)
    .agg(
        rows_utility_sheet=('Phase Name', 'size'),
        rows_with_capacity_ge_1=('capacity_ge_1', 'sum'),
        capacity_mw_utility_sheet=('capacity_mw_for_sum', 'sum'),
        countries_utility_sheet=('Country/Area', 'nunique'),
    )
    .sort_values(['status_group', 'rows_utility_sheet', 'Status'], ascending=[True, False, True])
)
status_scope_summary['capacity_mw_utility_sheet'] = status_scope_summary['capacity_mw_utility_sheet'].round(1)

country_filter_audit = pd.concat([
    clean_util.groupby('Country/Area').size().rename('record_count_clean'),
    clean_util.groupby('Country/Area')['Capacity (MW)'].sum().round(1).rename('capacity_mw_clean'),
], axis=1).reset_index().rename(columns={'Country/Area': 'country'})
country_filter_audit['passes_record_threshold'] = country_filter_audit['record_count_clean'] >= COUNTRY_THRESHOLD
country_filter_audit['threshold_gap_from_30'] = country_filter_audit['record_count_clean'] - COUNTRY_THRESHOLD
country_filter_audit['scope_group'] = np.where(country_filter_audit['passes_record_threshold'], 'retained', 'removed')
country_filter_audit = country_filter_audit.sort_values(['passes_record_threshold', 'record_count_clean', 'capacity_mw_clean', 'country'], ascending=[False, True, False, True])

status_profile = (
    retained.pivot_table(
        index='Country/Area',
        columns='Status',
        values='Phase Name',
        aggfunc='size',
        fill_value=0,
    )
    .reindex(columns=STATUS_ORDER, fill_value=0)
)

dominant_status = status_profile.idxmax(axis=1).rename('dominant_status')
country_scope_summary = pd.concat([
    retained.groupby('Country/Area').size().rename('record_count'),
    retained.groupby('Country/Area')['Capacity (MW)'].sum().round(1).rename('capacity_mw'),
    retained.groupby('Country/Area')['Capacity (MW)'].mean().round(2).rename('mean_capacity_mw'),
    dominant_status,
    status_profile.add_suffix('_rows'),
], axis=1).reset_index().rename(columns={'Country/Area': 'country'})
country_scope_summary = country_scope_summary.sort_values(['record_count', 'capacity_mw', 'country'], ascending=[False, False, True])

mode_activity_summary = (
    retained.assign(mode=retained['mode'].astype(str), capacity_bin=retained['capacity_bin'].astype(str))
    .groupby(['mode', 'Status', 'capacity_bin'], as_index=False)
    .agg(
        rows=('Phase Name', 'size'),
        capacity_mw=('Capacity (MW)', 'sum'),
        countries_active=('Country/Area', 'nunique'),
    )
)
full_mode_index = pd.DataFrame({
    'mode': MODE_ORDER,
    'Status': [status for status in STATUS_ORDER for _ in SIZE_BIN_LABELS],
    'capacity_bin': SIZE_BIN_LABELS * len(STATUS_ORDER),
})
mode_activity_summary = full_mode_index.merge(mode_activity_summary, how='left', on=['mode', 'Status', 'capacity_bin'])
mode_activity_summary[['rows', 'capacity_mw', 'countries_active']] = mode_activity_summary[['rows', 'capacity_mw', 'countries_active']].fillna(0)
mode_activity_summary['rows'] = mode_activity_summary['rows'].astype(int)
mode_activity_summary['countries_active'] = mode_activity_summary['countries_active'].astype(int)
mode_activity_summary['capacity_mw'] = mode_activity_summary['capacity_mw'].astype(float).round(1)
mode_activity_summary['country_coverage_pct'] = (100 * mode_activity_summary['countries_active'] / retained['Country/Area'].nunique()).round(2)
mode_activity_summary['row_share_pct'] = (100 * mode_activity_summary['rows'] / len(retained)).round(2)
mode_activity_summary['capacity_share_pct'] = (100 * mode_activity_summary['capacity_mw'] / retained_total_capacity).round(2)

bin_assignment_audit = (
    retained.assign(capacity_bin=retained['capacity_bin'].astype(str))
    .groupby('capacity_bin', as_index=False)
    .agg(
        rows=('Phase Name', 'size'),
        capacity_mw=('Capacity (MW)', 'sum'),
        countries_active=('Country/Area', 'nunique'),
        lower_bound_mw=('Capacity (MW)', 'min'),
        upper_bound_mw=('Capacity (MW)', 'max'),
    )
)
bin_assignment_audit['capacity_mw'] = bin_assignment_audit['capacity_mw'].round(1)
bin_assignment_audit['boundary_rule'] = [
    '1 <= MW < 5',
    '5 <= MW < 50',
    '50 <= MW < 500',
    'MW >= 500',
]

removed_rows_country_threshold = int(len(clean_util) - len(retained))
removed_capacity_country_threshold = float(clean_util['Capacity (MW)'].sum() - retained_total_capacity)
excluded_statuses = '; '.join(status_scope_summary.loc[status_scope_summary['status_group'] == 'excluded', 'Status'].tolist())

sample_summary = pd.DataFrame([
    {'metric': 'run_id', 'value': RUN_ID},
    {'metric': 'pipeline_version', 'value': PIPELINE_VERSION},
    {'metric': 'dataset_release', 'value': DATASET_RELEASE},
    {'metric': 'source_sheet', 'value': SHEET_NAME},
    {'metric': 'source_scope', 'value': 'utility-scale main sheet only'},
    {'metric': 'observational_unit', 'value': 'phase-level'},
    {'metric': 'distributed_sheet_excluded', 'value': True},
    {'metric': 'retained_statuses', 'value': '; '.join(STATUS_ORDER)},
    {'metric': 'excluded_statuses', 'value': excluded_statuses},
    {'metric': 'country_filter_rule', 'value': f'record_count >= {COUNTRY_THRESHOLD}'},
    {'metric': 'size_bins', 'value': '; '.join(SIZE_BIN_LABELS)},
    {'metric': 'bin_boundary_rule', 'value': '1 <= MW < 5; 5 <= MW < 50; 50 <= MW < 500; MW >= 500'},
    {'metric': 'countries_before_country_filter', 'value': int(clean_util['Country/Area'].nunique())},
    {'metric': 'countries_retained', 'value': int(retained['Country/Area'].nunique())},
    {'metric': 'countries_removed_by_country_filter', 'value': int(clean_util['Country/Area'].nunique() - retained['Country/Area'].nunique())},
    {'metric': 'rows_after_status_and_capacity_filters', 'value': int(len(clean_util))},
    {'metric': 'rows_retained', 'value': int(len(retained))},
    {'metric': 'rows_removed_by_country_filter', 'value': removed_rows_country_threshold},
    {'metric': 'capacity_mw_retained', 'value': round(retained_total_capacity, 1)},
    {'metric': 'capacity_mw_removed_by_country_filter', 'value': round(removed_capacity_country_threshold, 1)},
    {'metric': 'median_rows_per_country', 'value': float(round(retained.groupby('Country/Area').size().median(), 2))},
    {'metric': 'mean_rows_per_country', 'value': float(round(retained.groupby('Country/Area').size().mean(), 2))},
    {'metric': 'minimum_retained_country_record_count', 'value': int(retained.groupby('Country/Area').size().min())},
    {'metric': 'all_16_modes_assigned', 'value': bool(retained['mode'].nunique() == len(MODE_ORDER))},
    {'metric': 'unassigned_bin_count', 'value': int(retained['capacity_bin'].isna().sum())},
    {'metric': 'unassigned_mode_count', 'value': int(retained['mode'].isna().sum())},
])

retained.to_csv(TABLES / 'p1_retained_baseline.csv', index=False)
sample_summary.to_csv(TABLES / 'p1_retained_sample_summary.csv', index=False)
country_scope_summary.to_csv(TABLES / 'p1_country_scope_summary.csv', index=False)
mode_activity_summary.to_csv(TABLES / 'p1_mode_activity_summary.csv', index=False)
status_scope_summary.to_csv(TABLES / 'p1_status_scope_summary.csv', index=False)
country_filter_audit.to_csv(TABLES / 'p1_country_filter_audit.csv', index=False)
bin_assignment_audit.to_csv(TABLES / 'p1_bin_assignment_audit.csv', index=False)

threshold_edge = country_filter_audit.loc[country_filter_audit['record_count_clean'].between(25, 35)].sort_values(['record_count_clean', 'country'])
retained_status_rows = status_scope_summary.loc[status_scope_summary['status_group'] == 'retained', 'rows_utility_sheet'].sum()
excluded_status_rows = status_scope_summary.loc[status_scope_summary['status_group'] == 'excluded', 'rows_utility_sheet'].sum()

report_lines = [
    '# P1 Retained Scope Report',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    f'- dataset_release: `{DATASET_RELEASE}`',
    f'- sheet_name: `{SHEET_NAME}`',
    f'- source_scope: utility-scale main sheet only',
    f'- observational_unit: phase-level',
    '',
    '## Frozen analytical rules applied',
    f'- retained_statuses: {", ".join(STATUS_ORDER)}',
    f'- excluded_statuses: {excluded_statuses}',
    f'- country_filter_rule: record count >= {COUNTRY_THRESHOLD}',
    f'- size_bins: {", ".join(SIZE_BIN_LABELS)}',
    f'- bin_boundary_rule: 1 <= MW < 5; 5 <= MW < 50; 50 <= MW < 500; MW >= 500',
    f'- distributed_sheet_excluded_from_main_scope: yes',
    '',
    '## Retained analytical scope',
    f'- countries_retained: {retained["Country/Area"].nunique()}',
    f'- rows_retained: {len(retained):,}',
    f'- capacity_mw_retained: {retained_total_capacity:,.1f}',
    f'- assigned_modes: {retained["mode"].nunique()} / {len(MODE_ORDER)}',
    '',
    '## Status audit',
    f'- retained_status_rows_on_utility_sheet: {int(retained_status_rows):,}',
    f'- excluded_status_rows_on_utility_sheet: {int(excluded_status_rows):,}',
]
for row in status_scope_summary.itertuples(index=False):
    report_lines.append(
        f'- status={row.Status}; group={row.status_group}; rows_utility_sheet={int(row.rows_utility_sheet):,}; rows_with_capacity_ge_1={int(row.rows_with_capacity_ge_1):,}; capacity_mw_utility_sheet={float(row.capacity_mw_utility_sheet):,.1f}'
    )
report_lines.extend([
    '',
    '## Country filter audit',
    f'- countries_before_country_filter: {clean_util["Country/Area"].nunique()}',
    f'- countries_removed_by_country_filter: {clean_util["Country/Area"].nunique() - retained["Country/Area"].nunique()}',
    f'- rows_removed_by_country_filter: {removed_rows_country_threshold:,}',
    f'- capacity_mw_removed_by_country_filter: {removed_capacity_country_threshold:,.1f}',
    '- threshold_edge_countries_25_to_35_rows:',
])
for row in threshold_edge.itertuples(index=False):
    report_lines.append(
        f'  - {row.country}: record_count_clean={int(row.record_count_clean)}, passes_threshold={bool(row.passes_record_threshold)}, capacity_mw_clean={float(row.capacity_mw_clean):,.1f}'
    )
report_lines.extend([
    '',
    '## Bin assignment audit',
    f'- unassigned_bin_count: {int(retained["capacity_bin"].isna().sum())}',
    f'- unassigned_mode_count: {int(retained["mode"].isna().sum())}',
])
for row in bin_assignment_audit.itertuples(index=False):
    report_lines.append(
        f'- bin={row.capacity_bin}; rows={int(row.rows):,}; capacity_mw={float(row.capacity_mw):,.1f}; countries_active={int(row.countries_active)}; observed_range={float(row.lower_bound_mw):,.3f} to {float(row.upper_bound_mw):,.3f}; rule={row.boundary_rule}'
    )
report_lines.extend([
    '',
    '## Top countries by retained record count',
])
for row in country_scope_summary.head(10).itertuples(index=False):
    report_lines.append(
        f'- {row.country}: {int(row.record_count):,} rows; {float(row.capacity_mw):,.1f} MW; dominant_status={row.dominant_status}'
    )
report_lines.extend([
    '',
    '## T1 and SI.1 readiness',
    '- `p1_retained_sample_summary.csv` provides dataset release, source sheet, observational unit, retained and excluded statuses, country filter rule, size-bin rule, retained counts, and distributed-sheet exclusion.',
    '- `p1_country_scope_summary.csv` provides the retained country-level scope inputs for main-text `T1` and `SI.1`.',
    '- `p1_status_scope_summary.csv`, `p1_country_filter_audit.csv`, and `p1_bin_assignment_audit.csv` provide the QA/QC traceability needed to explain what was retained and what was excluded.',
])
(REPORTS / 'p1_retained_scope_report.md').write_text('\n'.join(report_lines), encoding='utf-8')

display(sample_summary)
display(status_scope_summary)
display(country_filter_audit.head(15))
display(country_scope_summary.head(10))
display(mode_activity_summary)
display(bin_assignment_audit)

display(Markdown('**Saved outputs**'))
display(pd.DataFrame({'path': [
    str(TABLES / 'p1_retained_baseline.csv'),
    str(TABLES / 'p1_retained_sample_summary.csv'),
    str(TABLES / 'p1_country_scope_summary.csv'),
    str(TABLES / 'p1_mode_activity_summary.csv'),
    str(TABLES / 'p1_status_scope_summary.csv'),
    str(TABLES / 'p1_country_filter_audit.csv'),
    str(TABLES / 'p1_bin_assignment_audit.csv'),
    str(REPORTS / 'p1_retained_scope_report.md'),
    str(RUNTIME / 'p1_retained_scope_manifest.json'),
]}))
